# 🪕 KORA-0.01 — Notebook 1 : les données

**GPU Challenge 2026 · Epitech Bénin**

Un modèle d'IA n'apprend que ce qu'on lui montre. Ce notebook fabrique **le jeu de données** de KORA : des milliers de messages, chacun avec son **étiquette** (sa catégorie).

| Catégorie | Exemple |
|---|---|
| `legitime` | « Vous avez reçu 15 000 FCFA de KOFFI… » (le vrai SMS officiel) |
| `demande_code` | « Ici le service client MTN, envoyez le code reçu par SMS… » |
| `faux_depot` | « J'ai envoyé 20 000 sur ton numéro par erreur, renvoie stp… » |
| `faux_gain` | « Félicitations, vous avez gagné 500 000 F, payez 5 000 F de frais… » ou « Prêt rapide sans garantie, payez d'abord les frais de dossier… » |
| `compte_bloque` | « Votre compte MoMo est bloqué, cliquez ici… » |

**Ce que fait ce notebook :**
1. Il génère des messages en français à partir de **modèles** (des phrases à trous), avec des variations réalistes (abréviations, fautes, emojis…).
2. Il découpe les données en **train / validation / test** de façon honnête (voir l'étape 4).
3. Il traduit une partie des messages en **fon** (brouillon automatique, à faire corriger par des locuteurs natifs).
4. Il prépare les fichiers que l'**équipe finance** remplira avec de **vrais messages** récoltés en interview.

⏱ Durée : 2 min sans traduction, ~10 min avec la traduction fon (GPU).
👉 Menu **Exécution → Tout exécuter**, ou exécute les cellules une par une avec ▶ (Maj + Entrée).

In [ ]:
# ▶ Connexion à Google Drive et chemins du projet
# En Colab : le dossier KORA doit être dans "Mon Drive" (MyDrive/KORA).
# En local : le notebook utilise le dossier parent.
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/KORA"
else:
    BASE = os.path.abspath("..")
sys.path.append(BASE)
for d in ["data", "modeles", "resultats"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("Dossier du projet :", BASE)
print("Contenu :", sorted(os.listdir(BASE)))
from kora_lib import CATEGORIES, NOMS, normaliser, detecter_signaux

## 1. Les ingrédients

Au lieu d'écrire 3 000 messages à la main, on écrit des **phrases à trous** comme :

> `Vous avez reçu {montant} FCFA de {nom}. Nouveau solde : {solde} FCFA.`

Puis on remplit les trous au hasard avec des valeurs réalistes. Cette cellule définit ces valeurs : opérateurs béninois, prénoms, montants, faux numéros, faux liens.

🔐 **Aucune donnée réelle n'est utilisée ici** : les numéros et les liens sont inventés.

In [ ]:
import random

rng = random.Random(42)   # graine fixe = mêmes résultats à chaque exécution (reproductible)

OPERATEURS = ["MTN MoMo", "MTN", "Moov Money", "Moov", "Celtiis Cash", "Celtiis"]
BANQUES = ["Ecobank", "BOA", "Orabank", "UBA", "NSIA Banque", "Coris Bank"]
PRENOMS = ["Koffi", "Ablawa", "Sèna", "Mahougnon", "Codjo", "Afiavi", "Rodrigue", "Fifamè", "Sêmèvo",
           "Gildas", "Rachidatou", "Ulrich", "Pélagie", "Romaric", "Bénédicta", "Ange", "Josué", "Grâce",
           "Firmin", "Nadège", "Yao", "Akim", "Inès", "Hervé"]
NOMS_FAMILLE = ["DOSSOU", "HOUNKPATIN", "AHOUANDJINOU", "AGBO", "ZINSOU", "KPADONOU", "SOSSOU",
                "TOSSOU", "GBAGUIDI", "ADJOVI", "HOUNSOU", "AKPLOGAN", "BIO", "SALIFOU", "OROU"]
ENTREPRISES = ["SBEE", "SONEB", "Canal+", "Jumia", "Gozem", "la Loterie Nationale", "Orange", "Total"]
DOMAINES = ["momo-bonus-bj.xyz", "mtn-verif.site", "moov-securite.top", "bj-promo.online",
            "celtiis-cadeau.click", "verif-compte-bj.info", "bonus-mobile.link", "bit.ly/3kR9xQ"]


def montant():
    v = rng.choice([500, 1000, 2000, 2500, 5000, 7500, 10000, 15000, 20000, 25000, 30000, 50000, 75000, 100000, 150000, 250000])
    style = rng.random()
    if style < 0.4:
        return f"{v:,}".replace(",", " ")      # 15 000
    if style < 0.6:
        return f"{v:,}".replace(",", ".")      # 15.000
    if style < 0.8:
        return str(v)                           # 15000
    return f"{v//1000}k" if v >= 1000 else str(v)   # 15k


def numero():
    # Format béninois à 10 chiffres (01 + 8 chiffres). Numéros FICTIFS, tirés au hasard.
    n = "01" + "".join(str(rng.randint(0, 9)) for _ in range(8))
    style = rng.random()
    if style < 0.4:
        return " ".join([n[i:i+2] for i in range(0, 10, 2)])
    if style < 0.7:
        return "+229 " + n
    return n


def valeurs():
    """Un jeu de valeurs tirées au hasard pour remplir UN message."""
    p = rng.choice(PRENOMS)
    return {
        "op": rng.choice(OPERATEURS), "banque": rng.choice(BANQUES), "entreprise": rng.choice(ENTREPRISES),
        "prenom": p, "nom": f"{rng.choice(NOMS_FAMILLE)} {p}",
        "montant": montant(), "montant2": montant(), "solde": montant(), "gain": montant(), "frais": montant(),
        "num": numero(), "code": str(rng.randint(1000, 999999)),
        "id": "".join(rng.choice("0123456789ABCDEF") for _ in range(10)),
        "lien": ("https://" if rng.random() < 0.5 else "") + rng.choice(DOMAINES),
        "ussd": rng.choice(["*880", "*155", "*400", "*123", "*889"]),
        "heure": f"{rng.randint(6, 22)}h{rng.choice(['00', '15', '30', '45'])}",
        "date": f"{rng.randint(1, 28):02d}/{rng.randint(1, 12):02d}",
        "carte": str(rng.randint(1000, 9999)),
    }

print(valeurs())

## 2. Les phrases à trous (modèles de messages)

Chaque catégorie a 20 à 50 modèles, et chaque type de message existe sous **plusieurs formulations** : ainsi, même si une formulation part dans le test, le modèle a vu les autres. Ils mélangent :
- des **SMS** (écrits),
- des **transcriptions d'appels** (ce qu'un arnaqueur dit au téléphone, car KORA analysera aussi les vocaux).

⚠️ **Les « pièges » sont volontaires dans `legitime`** : le vrai SMS officiel qui t'**envoie** un code (« Votre code est 4821, ne le partagez pas ») contient le mot *code*, comme les arnaques. Le modèle doit apprendre la différence entre **recevoir** un code (normal) et **devoir le donner** (arnaque). Sans ces exemples, il crierait à l'arnaque dès qu'il voit le mot « code ».

💡 **L'équipe finance peut ajouter des modèles ici** à partir des arnaques entendues en interview.

In [ ]:
MODELES = {
"legitime": [
    "Vous avez reçu {montant} FCFA de {nom} ({num}). Nouveau solde : {solde} FCFA. ID transaction : {id}.",
    "Transfert de {montant} FCFA vers {nom} effectué avec succès. Frais : 0 FCFA. Solde : {solde} FCFA. Réf {id}.",
    "Votre code de confirmation {op} est {code}. Ne le communiquez à personne, même à un agent {op}.",
    "Retrait de {montant} FCFA effectué chez l'agent {nom}. Nouveau solde {solde} FCFA. Merci d'utiliser {op}.",
    "Paiement de {montant} FCFA à {entreprise} réussi. Référence {id}. Conservez ce message.",
    "{op} : Profitez de 1 Go à 500F, tapez {ussd}#. Rappel : {op} ne vous demandera jamais votre code secret.",
    "{op} : forfait internet 2 Go à 1000F valable 7 jours. Pour souscrire, composez {ussd}#.",
    "Bonus {op} : vous avez reçu 100 Mo offerts pour votre recharge de {montant} F. Valable 24h. Aucune action requise.",
    "{op} vous remercie pour votre fidélité. Profitez de -20% sur les appels ce week-end en composant {ussd}#.",
    "Votre recharge de {montant} F a été effectuée avec succès. Votre bonus de crédit est disponible jusqu'à minuit.",
    "{op} : nouveau pass nuit, 1,5 Go de 23h à 6h pour 200F. Tapez {ussd}# pour l'activer.",
    "{op} Promo : transférez de l'argent sans frais ce vendredi. Offre valable dans l'application et via {ussd}#.",
    "{prenom} je t'ai envoyé {montant} pour le marché, confirme moi quand tu reçois stp",
    "Maman c'est {prenom}, j'ai fait le dépôt de {montant} ce matin, regarde ton solde quand tu peux",
    "Rappel tontine : rencontre samedi à {heure}, chacun apporte {montant} F. Merci à tous",
    "{banque} : votre carte se terminant par {carte} a été débitée de {montant} FCFA le {date}.",
    "Le code du wifi de la maison c'est {code}, ne le donne pas aux voisins hein",
    "Allô {prenom}, c'est moi. Je t'ai fait un transfert de {montant} pour le loyer, dis moi si c'est arrivé.",
    "Merci pour votre paiement de {montant} F. Votre commande sera livrée demain avant {heure}.",
    "Bonjour {prenom}, la réunion est déplacée à {heure}. Apporte les documents s'il te plaît.",
    "{code} est votre code de validation {op}. Valable 5 minutes. Ne le partagez jamais.",
    "Code OTP {banque} : {code}. Si vous n'êtes pas à l'origine de cette demande, ignorez ce message.",
    "{op} : transfert reçu de {montant}F du {num} ({nom}). Solde : {solde}F. Trx ID {id}",
    "Dépôt de {montant} FCFA reçu de l'agent {nom}. Nouveau solde {solde} FCFA.",
    "Vous avez envoyé {montant} FCFA à {nom} le {date} à {heure}. Solde disponible {solde} FCFA.",
    "Votre facture {entreprise} de {montant} F a été réglée. Merci de votre confiance.",
    "Achat de crédit de {montant} F réussi. Solde principal : {solde} F.",
    "{op} vous informe : maintenance prévue cette nuit de 1h à 3h. Merci de votre compréhension.",
    "Rappel {op} : ne communiquez jamais votre code secret, même à nos agents.",
    "{prenom} stp tu peux m'envoyer {montant} pour le transport ? Je te rembourse samedi.",
    "C'est bon j'ai reçu les {montant}, merci beaucoup {prenom} 🙏",
    "Allô {prenom}, tu es où ? On t'attend au marché depuis {heure}.",
    "Papa demande si tu as reçu l'argent de la scolarité, appelle-le ce soir.",
    "{banque} : virement de {montant} FCFA reçu sur votre compte le {date}. Solde {solde} FCFA.",
    "Allô oui c'est {prenom}, j'ai bien reçu le transfert, merci, on se voit demain.",
    "Le code pour ouvrir le portail c'est {code}, je suis encore au boulot.",
    "{banque} : votre demande de prêt est en cours d'étude. Un conseiller vous contactera. Aucun frais n'est demandé avant signature en agence.",
    "Rappel {banque} : votre échéance de prêt de {montant} FCFA sera prélevée le {date}. Merci de provisionner votre compte.",
    "Ne communiquez jamais votre code secret. {op} ne vous le demandera jamais par SMS ou par appel.",
    "Votre OTP {banque} est {code}. Ne le transmettez à personne. Valable 3 minutes.",
    "{op} : {code} est votre code pour confirmer le retrait de {montant} F. Si vous n'avez rien demandé, ignorez ce message.",
    "Vous avez acheté un forfait internet de {montant} F. Validité 30 jours. Merci de votre fidélité.",
    "Transaction réussie. Vous avez payé {montant} FCFA à {nom}. Nouveau solde : {solde} FCFA.",
    "{prenom}, l'argent pour l'école est parti, vérifie quand tu peux. Bisous.",
    "Bjr {prenom}, tu peux passer récupérer la commande demain ? Je t'envoie le reste à {heure}.",
    "Félicitations {prenom} pour ton diplôme ! Toute la famille est fière de toi 🎉",
    "Résultat du tirage de la tontine : c'est {prenom} qui prend ce mois-ci. Rendez-vous samedi à {heure}.",
    "Allô {prenom}, on a gagné le match ce matin, on fête ça ce soir ?",
    "{op} : votre compte a été réactivé avec succès. Merci de votre confiance.",
    "{banque} : votre nouvelle carte est disponible. Présentez votre pièce d'identité au guichet de votre agence.",
    "Promo {entreprise} : -20% sur tous les articles ce week-end, en magasin uniquement.",
    "Rappel : le loyer de {montant} F est attendu avant le 5 du mois. Merci.",
    "Bsr {prenom}, je t'ai renvoyé tes {montant} de la semaine dernière, merci encore pour le dépannage.",
    "Votre colis {entreprise} sera livré demain. Paiement à la livraison : {montant} F.",
],
"demande_code": [
    "Bonjour, ici le service client {op}. Suite à une mise à jour, envoyez le code reçu par SMS pour sécuriser votre compte.",
    "{op} : votre compte sera suspendu. Pour le maintenir, communiquez votre code secret au {num}.",
    "Bjr c'est l'agent {op}, j'ai fait une erreur sur ton compte, envoie moi le code que tu vas recevoir stp",
    "Allô, je suis l'agent {op}, on a remarqué une activité suspecte sur votre compte. Donnez-moi le code à 6 chiffres que vous venez de recevoir.",
    "Pour recevoir votre remboursement de {montant} FCFA, composez {ussd}*{num}*{montant}# et entrez votre code PIN.",
    "Monsieur, c'est le service technique {op}. Pour finaliser la migration de votre SIM, dites-moi votre code secret.",
    "Un client a fait un dépôt sur votre compte par erreur, pour l'annuler tapez {ussd}# puis votre code secret.",
    "Allô ma sœur, je suis conseiller {op}, ton compte a un problème. Il va t'arriver un code, lis-le moi vite.",
    "{op} INFO : vérification obligatoire. Répondez à ce message avec votre code PIN pour éviter la fermeture.",
    "Bonsoir, votre bonus fidélité de {montant} F est en attente. Pour le valider, transmettez le code OTP reçu au {num}.",
    "C'est {prenom} du service client, on doit réinitialiser ton compte. Tape {ussd}# et donne moi le code qui s'affiche.",
    "Allô oui, c'est la direction {op}. Votre numéro a été choisi pour une vérification. Le code que vous avez reçu, dites-le moi.",
    "Cher client {op}, votre numéro a été sélectionné pour une mise à jour. Merci de nous envoyer le code reçu par SMS au {num}.",
    "Allô, ici le service fraude de {banque}. Pour bloquer l'opération suspecte, donnez-moi le code que vous avez reçu.",
    "Svp c'est urgent, un code va arriver sur ton téléphone, envoie le moi c'est pour ma recharge",
    "Je suis ton cousin, j'ai perdu mon téléphone. Un code va arriver sur ton numéro, transfère-le moi vite.",
    "Composez {ussd}*1*{num}# puis saisissez votre code secret pour recevoir votre remboursement.",
    "Allô monsieur, c'est l'agence {op}, vous allez recevoir un SMS avec un code, il faut me le lire pour finaliser.",
    "Bonjour, je suis de la cellule anti-fraude {op}. Un code de sécurité vient de vous être envoyé, dictez-le moi pour annuler l'opération.",
    "Ton compte WhatsApp va être piraté, envoie-moi vite le code à 6 chiffres que tu vas recevoir pour le protéger.",
    "Ici l'agent qui a fait le dépôt chez vous, tapez {ussd}*1# et donnez-moi le code affiché pour corriger l'erreur.",
    "Pour finaliser votre inscription au programme {op} Plus, envoyez le code PIN de votre compte au {num}.",
],
"faux_depot": [
    "Bonsoir, j'ai envoyé {montant} sur ton numéro par erreur, c'était pour ma mère malade. Renvoie moi stp, que Dieu te bénisse.",
    "Vs avez recu {montant}F de {nom}. Votre solde actuelle est {solde}F. Veuillez retourner le montant si erreur.",
    "Je viens de te faire un dépôt de {montant}, c'est une erreur. Regarde ton SMS et renvoie {montant} au {num} vite",
    "Allô, excusez-moi madame, j'ai transféré {montant} sur votre numéro par erreur. S'il vous plaît, retournez-moi l'argent.",
    "Transfert reçu : {montant} FCFA de {nom}. Solde : {solde}. Ce dépôt est une erreur, merci de restituer la somme au {num}.",
    "Mon frère, je me suis trompé de numéro en envoyant {montant}. Renvoie juste sur ce numéro {num}, c'est urgent.",
    "Allô, vous avez reçu un message de dépôt de {montant}? C'était pour mon fils à l'école. Renvoyez s'il vous plaît, je vous en supplie.",
    "Dépôt effectué {montant}F sur votre compte. Erreur de transfert, veuillez rembourser immédiatement au {num} sinon plainte.",
    "Bonjour c'est l'agent {nom}, j'ai fait un dépôt de {montant} chez vous par erreur, renvoyez avant ce soir svp",
    "Je t'ai envoyé {montant} au lieu de {montant2}, renvoie moi la différence stp mon ami",
    "Votre compte a été crédité de {montant}F par erreur de transfert. Retournez le montant au {num} dans les 2h.",
    "Allô, c'est la cliente de ce matin, j'ai mis {montant} au lieu de {montant2} sur ton compte. Rembourse la différence vite.",
    "Bonjour, j'ai fait un retrait chez vous ce matin mais l'argent est venu sur votre numéro. Renvoyez {montant} svp.",
    "Vous avez reçu {montant} FCFA. Ce transfert est en attente : renvoyez {montant2} pour le confirmer.",
    "Ma sœur pardon, l'argent de la tontine est tombé chez toi par erreur, renvoie {montant} au {num}.",
    "TRANSFERT RECU {montant}F DE {nom}. SOLDE {solde}F. ATTENTION : TRANSACTION ERRONEE, RETOURNEZ LES FONDS AU {num}.",
    "Allô c'est le monsieur qui t'a envoyé {montant} tout à l'heure, je me suis trompé, renvoie s'il te plaît, je suis à l'hôpital.",
    "Bonsoir maman, c'est {prenom}, j'ai changé de numéro. J'ai envoyé {montant} chez toi par erreur, renvoie sur ce nouveau numéro {num}.",
    "Vous venez de recevoir {montant} FCFA. Solde : {solde}. Merci de reverser {montant2} au {num}, le surplus est une erreur du système.",
    "Allô tonton, j'ai fait un dépôt chez toi pour mon frère, c'était une erreur, renvoie-moi vite avant ce soir.",
],
"faux_gain": [
    "Félicitations ! Votre numéro a gagné {gain} FCFA au tirage {op}. Pour retirer, envoyez {frais} FCFA de frais au {num}.",
    "Promo {op} 🎉 vous êtes le grand gagnant d'un iPhone. Cliquez {lien} pour réclamer votre lot.",
    "Bonus spécial : recevez 3 fois votre dépôt aujourd'hui seulement. Envoyez {montant} au {num} et recevez le triple.",
    "Placement sûr : {montant} F aujourd'hui = {gain} F dans 48h. Places limitées, contacte vite {num}.",
    "Recrutement {entreprise} : vous êtes retenu. Payez {frais} F de frais de dossier au {num} pour valider votre poste.",
    "Allô, félicitations ! C'est {entreprise}, vous avez gagné {gain} F. Il faut juste payer {frais} F pour les frais de retrait.",
    "Jackpot {op} : votre numéro est sorti au tirage du mois. Gain de {gain} FCFA. Activez votre lot ici : {lien}",
    "Tu veux doubler ton argent ? Envoie {montant} et tu reçois le double en 30 minutes, c'est garanti mon frère.",
    "Cadeau de fin d'année {op} : {gain} F offerts à 100 clients. Confirmez avec {frais} F de frais d'activation.",
    "Félicitations, vous avez été sélectionné pour une bourse de {gain} F. Frais d'inscription : {frais} F au {num}.",
    "Allô, vous avez gagné une moto au jeu {op}. Pour la livraison, envoyez {frais} F au livreur au {num}.",
    "Investissement crypto : {montant} F placés = {gain} F par semaine. Rejoins le groupe vite : {lien}",
    "Vous avez gagné un bon d'achat de {gain} F chez {entreprise}. Validez avec {frais} F au {num}.",
    "Opportunité : gagne {gain} F par jour depuis ton téléphone. Inscription {frais} F seulement.",
    "Allô, c'est la radio, vous avez gagné notre jeu ! Envoyez {frais} F pour les frais de transfert de votre lot.",
    "Programme d'aide du gouvernement : {gain} F pour les commerçants. Inscris-toi avec {frais} F au {num}.",
    "Tontine en ligne : mets {montant} F, tu reçois {gain} F dans 7 jours. 100% sûr.",
    "Prêt rapide sans garantie : recevez {gain} F en 10 minutes sur votre MoMo. Payez d'abord {frais} F de frais de dossier au {num}.",
    "Félicitations, votre demande de crédit de {gain} FCFA est approuvée ! Envoyez {frais} F pour l'assurance du prêt avant le décaissement.",
    "Besoin d'argent urgent ? Crédit de {gain} F sans justificatif, remboursable en 6 mois. Activation : {frais} F au {num}.",
    "{banque} Microfinance : votre prêt de {gain} F est disponible. Validez votre dossier sur {lien} avec votre code secret.",
    "Allô, c'est le service crédit de {op}. Votre prêt de {gain} francs est prêt, il faut juste envoyer {frais} francs pour le timbre.",
    "Prêt aux commerçantes sans intérêt : {gain} F accordés. Frais d'inscription {frais} F. Places limitées, réponds vite.",
    "Bravo ! Vous faites partie des 50 gagnants du quiz {entreprise}. Pour recevoir {gain} F, payez {frais} F de frais de transfert.",
    "Gagne {gain} F par semaine en likant des vidéos TikTok. Inscription {frais} F. Écris au {num}.",
    "Héritage : un parent éloigné vous a légué {gain} FCFA. Frais de notaire de {frais} F à envoyer au {num}.",
],
"compte_bloque": [
    "{op} : votre compte a été bloqué pour activité suspecte. Cliquez sur {lien} pour le réactiver.",
    "Votre SIM sera désactivée dans 24h. Mettez à jour vos informations : {lien}",
    "Alerte sécurité {banque} : connexion inhabituelle détectée. Vérifiez votre compte ici {lien}",
    "Téléchargez la nouvelle application {op} Plus pour éviter la suspension de votre compte : {lien}",
    "Envoyez une photo de votre pièce d'identité au {num} sur WhatsApp pour débloquer votre compte.",
    "Allô, c'est {op}, votre numéro va être suspendu ce soir. Il faut confirmer votre identité, donnez votre date de naissance et votre numéro de CNI.",
    "Dernier rappel : votre compte {op} expire aujourd'hui. Renouvelez gratuitement sur {lien}",
    "{banque} : votre carte a été bloquée. Pour la débloquer, connectez-vous sur {lien} avec vos identifiants.",
    "Votre compte mobile money est suspendu. Contactez immédiatement le service au {num} pour la réactivation.",
    "Allô madame, je suis technicien {op}. Votre SIM a un problème, il faut installer l'application que je vais vous envoyer.",
    "Mise à jour obligatoire de votre compte {op}. Sans action sous 12h, votre argent sera bloqué : {lien}",
    "Vérification KYC {op} : envoyez votre selfie avec votre carte d'identité au {num} avant ce soir.",
    "Votre compte {op} sera clôturé pour défaut d'identification. Envoyez votre CNI au {num}.",
    "Cher client {banque}, mettez à jour vos coordonnées bancaires pour éviter le blocage : {lien}",
    "{op} : activité frauduleuse sur votre ligne. Installez l'application de sécurité : {lien}",
    "Votre numéro sera désactivé suite à la nouvelle loi d'identification. Envoyez nom, date de naissance et photo au {num}.",
    "Allô, service sécurité {op}, votre compte a été piraté. Il faut vous connecter tout de suite sur le site que je vous envoie.",
    "Votre ligne {op} sera coupée ce soir pour défaut de mise à jour. Composez {ussd}# et saisissez votre code secret.",
    "Alerte : tentative de connexion sur votre compte {op} depuis l'étranger. Sécurisez-le ici : {lien}",
    "{banque} : votre compte est gelé suite à un contrôle. Envoyez une copie de votre CNI et un selfie au {num}.",
],
}

for cat, liste in MODELES.items():
    print(f"{cat:15s} {len(liste):3d} modèles")

## 3. Les variations : rendre les messages réalistes

Les vrais SMS ne sont pas parfaits : abréviations (`vs`, `stp`, `bjr`), pas d'accents, majuscules, emojis, fautes de frappe… Si on n'entraîne qu'avec des phrases propres, le modèle sera perdu face aux vrais messages.

Cette fonction applique ces « défauts » au hasard.

In [ ]:
import re
from kora_lib import enlever_tons

ABREVIATIONS = [("s'il vous plaît", "svp"), ("s'il te plaît", "stp"), ("vous", "vs"), ("bonjour", "bjr"),
                ("bonsoir", "bsr"), ("merci", "mrc"), ("c'est", "c"), ("aujourd'hui", "ajd"),
                ("beaucoup", "bcp"), ("numéro", "num"), ("message", "msg"), ("quelque", "qlq"), ("pour", "pr")]
EMOJIS = ["🙏", "😊", "‼️", "🎉", "💰", "⚠️", "📲", "😭", "🔥"]


def faute_de_frappe(texte):
    mots = texte.split()
    i = rng.randrange(len(mots))
    m = mots[i]
    if len(m) > 3:
        j = rng.randrange(len(m) - 1)
        mots[i] = m[:j] + m[j+1] + m[j] + m[j+2:]   # inverse deux lettres voisines
    return " ".join(mots)


def varier(texte):
    t = texte
    if rng.random() < 0.35:
        for long, court in ABREVIATIONS:
            if rng.random() < 0.6:
                t = re.sub(rf"\b{re.escape(long)}\b", court, t, flags=re.I)
    if rng.random() < 0.30:
        t = enlever_tons(t)                      # beaucoup de SMS sont écrits sans accents
    r = rng.random()
    if r < 0.10:
        t = t.upper()
    elif r < 0.35:
        t = t.lower()
    if rng.random() < 0.20:
        t = t + " " + rng.choice(EMOJIS)
    if rng.random() < 0.12:
        t = faute_de_frappe(t)
    if rng.random() < 0.10:
        t = t.replace(".", "").replace(",", "")
    return t

print(varier("Bonjour, s'il vous plaît envoyez le code reçu pour sécuriser votre compte."))
print(varier("Bonjour, s'il vous plaît envoyez le code reçu pour sécuriser votre compte."))

## 4. Génération et découpage honnête train / validation / test

On fabrique environ **40 messages par modèle**.

### ⚠️ Le point le plus important du projet (à savoir expliquer au jury)

Si on mélangeait tous les messages au hasard, le **test** contiendrait des messages fabriqués à partir des **mêmes phrases à trous** que l'entraînement. Le modèle aurait l'air excellent… simplement parce qu'il reconnaît la phrase à trous. C'est de la **triche involontaire** (on appelle ça une *fuite de données*).

👉 On découpe donc **par modèle de phrase** : les phrases à trous du test ne sont **jamais vues** à l'entraînement. Le score sur le test mesure vraiment si KORA reconnaît **une arnaque qu'il n'a jamais vue**.

| Partie | Rôle |
|---|---|
| **train** (~65 %) | le modèle apprend dessus |
| **val** (~15 %) | on surveille l'apprentissage, on choisit le meilleur moment pour s'arrêter |
| **test** (~20 %) | l'examen final, utilisé **une seule fois** à la fin |

In [ ]:
import math
import pandas as pd

N_PAR_MODELE = 40
lignes = []
for cat, liste in MODELES.items():
    ids = list(range(len(liste)))
    rng.shuffle(ids)
    n_test = math.ceil(0.20 * len(ids))
    n_val = math.ceil(0.15 * len(ids))
    repartition = {i: "test" for i in ids[:n_test]}
    repartition.update({i: "val" for i in ids[n_test:n_test + n_val]})
    repartition.update({i: "train" for i in ids[n_test + n_val:]})
    for i, modele in enumerate(liste):
        for _ in range(N_PAR_MODELE):
            texte = varier(modele.format(**valeurs()))
            lignes.append({"texte": texte, "categorie": cat, "modele_id": f"{cat}_{i:02d}",
                           "split": repartition[i], "langue": "fr", "source": "synthetique"})

df = pd.DataFrame(lignes).drop_duplicates(subset="texte").reset_index(drop=True)
print("Nombre total de messages :", len(df))
print()
print(pd.crosstab(df["categorie"], df["split"], margins=True, margins_name="TOTAL"))

In [ ]:
# Vérification : aucun modèle de phrase ne se retrouve dans deux parties à la fois
fuites = df.groupby("modele_id")["split"].nunique()
assert (fuites == 1).all(), "Un modèle de phrase est présent dans plusieurs parties !"
print("✅ Aucune fuite : chaque phrase à trous appartient à une seule partie.")

# Quelques exemples au hasard
pd.set_option("display.max_colwidth", 140)
df.sample(10, random_state=1)[["categorie", "split", "texte"]]

In [ ]:
# Sauvegarde sur le Drive
df.to_csv(f"{BASE}/data/kora_fr.csv", index=False)
print("💾 Sauvegardé :", f"{BASE}/data/kora_fr.csv")

## 5. Le fon 🇧🇯

KORA doit comprendre le fon, surtout pour les **appels** (transcrits par le modèle `mms-300m-fongbe` dans le notebook 3).

Il n'existe presque pas de messages d'arnaque écrits en fon. On procède en deux temps :

**a) Brouillon automatique (pour l'entraînement).** On traduit ~400 messages du *train* avec **NLLB-200** de Meta, un modèle de traduction qui connaît le fon (`fon_Latn`). La qualité est moyenne : c'est un **brouillon**. Il sert seulement à l'entraînement.
⚠️ Licence de NLLB : CC-BY-NC (non commercial). OK pour le challenge ; pour un produit commercial, il faudra remplacer ces brouillons par des traductions humaines.

**b) Traduction humaine (pour le test).** On exporte 50 messages du **test** dans `a_traduire_en_fon.csv`. **Des locuteurs natifs de l'équipe ou de l'entourage les traduisent.** C'est ce fichier qui servira à mesurer honnêtement KORA en fon.

⏱ La traduction automatique nécessite le GPU (Exécution → Modifier le type d'exécution → T4 GPU).

In [ ]:
TRADUIRE_FON = True     # mets False pour sauter cette étape
NB_A_TRADUIRE = 400

try:
    import torch
    GPU = torch.cuda.is_available()
except ImportError:
    GPU = False

if TRADUIRE_FON and GPU:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    nom_modele = "facebook/nllb-200-distilled-600M"
    tok = AutoTokenizer.from_pretrained(nom_modele, src_lang="fra_Latn")
    traducteur = AutoModelForSeq2SeqLM.from_pretrained(nom_modele, torch_dtype=torch.float16).to("cuda")

    a_traduire = df[df["split"] == "train"].groupby("categorie").sample(n=NB_A_TRADUIRE // len(MODELES), random_state=0)
    textes = a_traduire["texte"].tolist()
    traductions = []
    for i in range(0, len(textes), 32):
        lot = textes[i:i+32]
        entree = tok(lot, return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
        with torch.no_grad():
            sortie = traducteur.generate(**entree, forced_bos_token_id=tok.convert_tokens_to_ids("fon_Latn"),
                                         max_new_tokens=128, num_beams=2)
        traductions += tok.batch_decode(sortie, skip_special_tokens=True)
        print(f"  {len(traductions)}/{len(textes)} traduits", end="\r")

    fon = a_traduire.copy()
    fon["texte_fr"] = fon["texte"]
    fon["texte"] = traductions
    fon["langue"] = "fon"
    fon["source"] = "nllb_brouillon"
    fon.to_csv(f"{BASE}/data/kora_fon_brouillon.csv", index=False)
    print("\n💾 Sauvegardé :", f"{BASE}/data/kora_fon_brouillon.csv")
    display(fon[["categorie", "texte_fr", "texte"]].head(8))
    del traducteur; torch.cuda.empty_cache()
else:
    print("⏭ Traduction fon sautée (pas de GPU ou TRADUIRE_FON = False).")

In [ ]:
# b) Le fichier que les locuteurs natifs vont traduire (pris dans le TEST : jamais vu à l'entraînement)
chemin = f"{BASE}/data/a_traduire_en_fon.csv"
if not os.path.exists(chemin):
    test = df[df["split"] == "test"]
    a_faire = test.groupby("categorie").sample(n=10, random_state=0)
    a_faire = a_faire[["categorie", "texte"]].rename(columns={"texte": "texte_fr"})
    a_faire["texte_fon"] = ""     # ← à remplir par un locuteur natif
    a_faire.to_csv(chemin, index=False)
    print("📝 Créé :", chemin)
else:
    print("📝 Déjà présent (on ne l'écrase pas) :", chemin)
faits = pd.read_csv(chemin).fillna("")
print(f"   {int((faits['texte_fon'].str.strip() != '').sum())}/{len(faits)} messages déjà traduits en fon")

## 6. Les vrais messages (équipe finance) 🤝

Le meilleur test de KORA : de **vrais** messages d'arnaque reçus par de vraies personnes au Bénin.

Pendant les interviews, l'équipe finance remplit `data/vrais_messages.csv` avec ces colonnes :

| colonne | contenu |
|---|---|
| `texte` | le message, recopié (⚠️ **remplacer** les numéros, noms et codes réels par des X) |
| `categorie` | une des 5 catégories : `legitime`, `demande_code`, `faux_depot`, `faux_gain`, `compte_bloque` |
| `langue` | `fr`, `fon` ou `mixte` |
| `canal` | `sms`, `appel`, `whatsapp`… |

🔐 **Règles** (venant du document finance) : consentement de la personne, **jamais** de PIN, d'OTP, de numéro complet ni de pièce d'identité.

Ces messages ne servent **qu'au test** : le modèle ne les voit jamais à l'entraînement. Pense aussi à récolter de **vrais messages légitimes** (SMS officiels de MTN/Moov), sinon on ne peut pas mesurer les fausses alertes.

In [ ]:
modele_vrais = f"{BASE}/data/vrais_messages_MODELE.csv"
pd.DataFrame([
    {"texte": "Bonjour je suis agent MTN, envoyez le code que vous allez recevoir. Mon numero XX XX XX XX", "categorie": "demande_code", "langue": "fr", "canal": "appel"},
    {"texte": "Vous avez recu 25000 FCFA de XXXX. Nouveau solde: XXXX FCFA", "categorie": "legitime", "langue": "fr", "canal": "sms"},
]).to_csv(modele_vrais, index=False)
print("📄 Modèle de fichier pour l'équipe finance :", modele_vrais)

vrais = f"{BASE}/data/vrais_messages.csv"
if os.path.exists(vrais):
    v = pd.read_csv(vrais)
    inconnues = set(v["categorie"]) - set(CATEGORIES)
    print(f"✅ {len(v)} vrais messages trouvés.", "⚠️ Catégories inconnues : " + str(inconnues) if inconnues else "")
    print(v["categorie"].value_counts())
else:
    print("⏳ Pas encore de vrais_messages.csv : copie le modèle, remplis-le, renomme-le en vrais_messages.csv.")

## ✅ Récapitulatif

Fichiers créés dans `KORA/data/` :
- `kora_fr.csv` : le jeu de données en français, découpé train / val / test **par modèle de phrase**
- `kora_fon_brouillon.csv` : brouillons fon pour l'entraînement (si le GPU était activé)
- `a_traduire_en_fon.csv` : 50 messages du test à **traduire en fon** par des natifs
- `vrais_messages_MODELE.csv` : le format pour les vrais messages récoltés par l'équipe finance

### 🎤 Ce que tu dis au jury
> « Nos données sont **synthétiques mais réalistes** : des modèles de messages tirés de vraies arnaques béninoises, avec les variations des vrais SMS. On a découpé **par modèle de phrase** pour éviter la fuite de données : notre score de test mesure la capacité à reconnaître des arnaques **jamais vues**. Et on valide sur de **vrais messages** récoltés sur le terrain. »

➡️ **Suite : Notebook 2 — le cerveau de KORA.**